# Daily dispatch optimisation

Linear program for daily battery scheduling with power, state-of-charge, efficiency and terminal-SoC constraints, solved with CVXPY.


## Formulation

For every hour \(t\), the optimiser chooses:

- \(c_t\): grid energy bought for charging, in MWh
- \(d_t\): grid energy sold by discharging, in MWh
- \(s_t\): stored battery energy, in MWh

The objective maximises contribution after the simplified wear allowance:

\[\max \sum_t [p_t(d_t-c_t)-k d_t]\]

Constraints enforce power limits, SoC bounds, energy accounting and the terminal SoC condition:

\[s_{t+1}=s_t+\eta_c c_t-\frac{d_t}{\eta_d}\]

\[0.2 \leq s_t \leq 1.8, \quad 0 \leq c_t,d_t \leq 1.0, \quad s_T=s_0\]

For this demonstration, the optimiser receives the realised price vector. This creates a **perfect-information reference**, not a deployable forecast-based strategy. Forecasts are added later.

In [7]:
from pathlib import Path

import cvxpy as cp
import numpy as np
import pandas as pd

prices = pd.read_csv(
    Path("data/belgium_day_ahead_sample.csv"),
    parse_dates=["timestamp"],
).sort_values("timestamp").reset_index(drop=True)

price_eur_per_mwh = prices["price_eur_per_mwh"].to_numpy(dtype=float)

CAPACITY_MWH = 2.0
POWER_LIMIT_MW = 1.0
MIN_SOC_MWH = 0.10 * CAPACITY_MWH
MAX_SOC_MWH = 0.90 * CAPACITY_MWH
CHARGE_EFFICIENCY = 0.92
DISCHARGE_EFFICIENCY = 0.92
WEAR_EUR_PER_MWH_DISCHARGED = 5.0
STEP_HOURS = 1.0
INITIAL_SOC_MWH = 0.8

prices.head()


,timestamp,price_eur_per_mwh
0,2025-01-06 00:00:00+01:00,61.2
1,2025-01-06 01:00:00+01:00,57.8
2,2025-01-06 02:00:00+01:00,52.1
3,2025-01-06 03:00:00+01:00,48.7
4,2025-01-06 04:00:00+01:00,45.9


## Build the optimisation problem


In [8]:
number_of_hours = len(prices)

# Grid-side energy decisions for each hourly interval.
charge_mwh = cp.Variable(number_of_hours, nonneg=True)
discharge_mwh = cp.Variable(number_of_hours, nonneg=True)

# State of charge is energy inside the battery after each interval.
soc_mwh = INITIAL_SOC_MWH + cp.cumsum(
    CHARGE_EFFICIENCY * charge_mwh - discharge_mwh / DISCHARGE_EFFICIENCY
)

constraints = [
    charge_mwh <= POWER_LIMIT_MW * STEP_HOURS,
    discharge_mwh <= POWER_LIMIT_MW * STEP_HOURS,
    soc_mwh >= MIN_SOC_MWH,
    soc_mwh <= MAX_SOC_MWH,
    soc_mwh[-1] == INITIAL_SOC_MWH,
]

sales_revenue = price_eur_per_mwh @ discharge_mwh
purchase_cost = price_eur_per_mwh @ charge_mwh
wear_cost = WEAR_EUR_PER_MWH_DISCHARGED * cp.sum(discharge_mwh)
contribution = sales_revenue - purchase_cost - wear_cost

problem = cp.Problem(cp.Maximize(contribution), constraints)


## Solve safely


In [9]:
optimal_contribution_eur = problem.solve(solver="CLARABEL")

if problem.status not in {"optimal", "optimal_inaccurate"}:
    raise RuntimeError(f"Optimisation failed with status: {problem.status}")

print(f"Solver status: {problem.status}")
print(f"Optimal contribution after modelled wear: {optimal_contribution_eur:.2f} EUR")


Solver status: optimal
Optimal contribution after modelled wear: 131.32 EUR


## Convert the solution into an auditable action log


In [10]:
optimised_log = prices.copy()
optimised_log["charge_mwh"] = charge_mwh.value
optimised_log["discharge_mwh"] = discharge_mwh.value
optimised_log["soc_mwh"] = soc_mwh.value
optimised_log["purchase_cost_eur"] = (
    optimised_log["price_eur_per_mwh"] * optimised_log["charge_mwh"]
)
optimised_log["sales_revenue_eur"] = (
    optimised_log["price_eur_per_mwh"] * optimised_log["discharge_mwh"]
)
optimised_log["wear_cost_eur"] = (
    WEAR_EUR_PER_MWH_DISCHARGED * optimised_log["discharge_mwh"]
)
optimised_log["contribution_eur"] = (
    optimised_log["sales_revenue_eur"]
    - optimised_log["purchase_cost_eur"]
    - optimised_log["wear_cost_eur"]
)

summary = pd.Series({
    "purchase_cost_eur": optimised_log["purchase_cost_eur"].sum(),
    "sales_revenue_eur": optimised_log["sales_revenue_eur"].sum(),
    "wear_cost_eur": optimised_log["wear_cost_eur"].sum(),
    "contribution_eur": optimised_log["contribution_eur"].sum(),
    "starting_soc_mwh": INITIAL_SOC_MWH,
    "ending_soc_mwh": optimised_log["soc_mwh"].iloc[-1],
})

summary.round(2)


purchase_cost_eur    184.99
sales_revenue_eur    331.03
wear_cost_eur         14.72
contribution_eur     131.32
starting_soc_mwh       0.80
ending_soc_mwh         0.80
dtype: float64

## Validate the solution


In [11]:
TOLERANCE = 1e-7

assert problem.status in {"optimal", "optimal_inaccurate"}
assert (optimised_log["charge_mwh"] >= -TOLERANCE).all()
assert (optimised_log["discharge_mwh"] >= -TOLERANCE).all()
assert (optimised_log["charge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + TOLERANCE).all()
assert (optimised_log["discharge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + TOLERANCE).all()
assert (optimised_log["soc_mwh"] >= MIN_SOC_MWH - TOLERANCE).all()
assert (optimised_log["soc_mwh"] <= MAX_SOC_MWH + TOLERANCE).all()
assert np.isclose(optimised_log["soc_mwh"].iloc[-1], INITIAL_SOC_MWH, atol=TOLERANCE)

simultaneous_energy_mwh = np.minimum(
    optimised_log["charge_mwh"], optimised_log["discharge_mwh"]
).max()
assert simultaneous_energy_mwh < TOLERANCE

print("All optimisation validation checks passed.")
print(f"Maximum simultaneous charge/discharge: {simultaneous_energy_mwh:.2e} MWh")


All optimisation validation checks passed.
Maximum simultaneous charge/discharge: 8.45e-12 MWh


## Inspect non-zero actions


In [12]:
action_mask = (optimised_log["charge_mwh"] > TOLERANCE) | (
    optimised_log["discharge_mwh"] > TOLERANCE
)

optimised_log.loc[
    action_mask,
    [
        "timestamp",
        "price_eur_per_mwh",
        "charge_mwh",
        "discharge_mwh",
        "soc_mwh",
        "contribution_eur",
    ],
].round(4)


,timestamp,price_eur_per_mwh,charge_mwh,discharge_mwh,soc_mwh,contribution_eur
3,2025-01-06 03:00:00+01:00,48.7,0.0870,0.000,0.880,-4.2348
4,2025-01-06 04:00:00+01:00,45.9,1.0000,0.000,1.800,-45.9000
7,2025-01-06 07:00:00+01:00,91.6,0.0000,0.472,1.287,40.8752
8,2025-01-06 08:00:00+01:00,104.2,0.0000,1.000,0.200,99.2000
13,2025-01-06 13:00:00+01:00,55.1,0.7391,0.000,0.880,-40.7261
14,2025-01-06 14:00:00+01:00,51.8,1.0000,0.000,1.800,-51.8000
18,2025-01-06 18:00:00+01:00,128.7,0.0000,1.000,0.713,123.7000
19,2025-01-06 19:00:00+01:00,116.3,0.0000,0.472,0.200,52.5336
23,2025-01-06 23:00:00+01:00,64.9,0.6522,0.000,0.800,-42.3261
